# Offline Showdex -> poke-engine MCTS -> Policy MLP Training

This notebook runs the full offline policy-predictor workflow: open or clone `showdown-trainer`, install dependencies, download the same pkmn preset data paths used by Showdex, collect Showdex-guided `poke-engine` MCTS targets into JSONL, run sanity checks, train the policy MLP, and verify the checkpoint.

No Pokemon Showdown login or websocket battle is required. After the Showdex/pkmn JSON files are cached locally, collection is offline-capable.

## 1. Locate Or Clone The Trainer Repo

If this notebook is already inside a local clone, the setup cell reuses it. In Colab, set `TRAINER_REPO_URL` to your fork before running.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

TRAINER_REPO_URL = "https://github.com/YOUR_USERNAME/showdown-trainer.git"  # edit for Colab
BRANCH = None  # e.g. "main" or your feature branch

def run(cmd, cwd=None):
    print("$", " ".join(str(part) for part in cmd))
    return subprocess.run([str(part) for part in cmd], cwd=cwd, check=True)

def looks_like_repo(path: Path) -> bool:
    return (path / "train.py").exists() and (path / "collect_offline_mcts.py").exists()

cwd = Path.cwd()
base_dir = Path("/content") if "google.colab" in sys.modules else cwd

if looks_like_repo(cwd):
    repo = cwd
else:
    repo = base_dir / "showdown-trainer"
    if not repo.exists():
        clone_cmd = ["git", "clone"]
        if BRANCH:
            clone_cmd += ["--branch", BRANCH]
        clone_cmd += [TRAINER_REPO_URL, repo]
        run(clone_cmd)

os.chdir(repo)
print(f"Using repo: {repo.resolve()}")


## 2. Install Python Dependencies

`poke-engine` provides the offline MCTS search. Colab already has PyTorch in most runtimes, but installing from `requirements.txt` keeps the notebook self-contained.

In [ ]:
run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])

import poke_engine
import torch

print("torch", torch.__version__, "cuda", torch.cuda.is_available())
print("poke_engine loaded")


## 3. Cache Showdex / pkmn Distributions

Showdex uses the pkmn repository under `https://pkmn.github.io`. This cell downloads the random battle preset and usage-stat JSON once. Reruns use the cached local files in `showdex_cache/`.

In [ ]:
import urllib.request

POKEMON_FORMAT = "gen9randombattle"
SHOWDEX_CACHE = Path("showdex_cache")
SHOWDEX_CACHE.mkdir(parents=True, exist_ok=True)

downloads = {
    SHOWDEX_CACHE / f"{POKEMON_FORMAT}.json": f"https://pkmn.github.io/randbats/data/{POKEMON_FORMAT}.json",
    SHOWDEX_CACHE / f"{POKEMON_FORMAT}-stats.json": f"https://pkmn.github.io/randbats/data/stats/{POKEMON_FORMAT}.json",
}

for path, url in downloads.items():
    if path.exists() and path.stat().st_size > 0:
        print(f"Using cached {path}")
        continue
    print(f"Downloading {url}")
    urllib.request.urlretrieve(url, path)
    print(f"Wrote {path} ({path.stat().st_size:,} bytes)")


## 4. Collect Offline MCTS Training Data

For a quick notebook verification, `SMOKE_RUN=True` keeps the search small. Increase `POSITIONS`, `HYPOTHESES`, and `MCTS_MS` for a larger training set.

In [ ]:
CONFIG_PATH = Path("configs/student.yaml")
DATA_PATH = Path("training_data/mcts_run.jsonl")
CHECKPOINT_PATH = Path("checkpoints/policy_mlp.pt")

SMOKE_RUN = True
POSITIONS = 8 if SMOKE_RUN else 64
HYPOTHESES = 2 if SMOKE_RUN else 4
MCTS_MS = 25 if SMOKE_RUN else 75

DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
if DATA_PATH.exists():
    DATA_PATH.unlink()

run([
    sys.executable,
    "collect_offline_mcts.py",
    "--config", CONFIG_PATH,
    "--positions", POSITIONS,
    "--hypotheses", HYPOTHESES,
    "--search-time-ms", MCTS_MS,
])

print(DATA_PATH, DATA_PATH.stat().st_size, "bytes")
print("rows", sum(1 for _ in DATA_PATH.open()))


## 5. Run Sanity Checks

In [ ]:
run([sys.executable, "sanity_check.py"])


## 6. Train The Policy MLP

The training script saves model and optimizer state, so the checkpoint can be resumed later with `training.resume_checkpoint_path` or `--resume-checkpoint-path`.

In [ ]:
train_cmd = [sys.executable, "train.py", "--config", CONFIG_PATH]
if SMOKE_RUN:
    train_cmd.append("--smoke")
run(train_cmd)


## 7. Verify The Checkpoint

In [ ]:
import json

from encode import EncoderConfig
from train import load_policy_checkpoint, predict_policy

model, checkpoint = load_policy_checkpoint(CHECKPOINT_PATH)
assert "model_state_dict" in checkpoint
assert "optimizer_state_dict" in checkpoint

first_decision = None
with DATA_PATH.open() as file:
    for line in file:
        record = json.loads(line)
        if record.get("record_type") == "decision":
            first_decision = record
            break

assert first_decision is not None
encoder_config = EncoderConfig(**checkpoint["encoder_config"])
probs = predict_policy(model, first_decision["state"], encoder_config=encoder_config)

print("checkpoint", CHECKPOINT_PATH.resolve())
print("saved update", checkpoint.get("update"), "saved epoch", checkpoint.get("epoch"))
print("prob sum", round(sum(probs), 6))
print("policy probs", [round(value, 4) for value in probs])


## 8. Train A Value Head From Trajectory Outcomes

Value targets need terminal game results. The offline position dataset at `DATA_PATH` from `collect_offline_mcts.py` contains MCTS policy targets but no winners, so this section expects a trajectory JSONL produced by `collect_trajectory_mcts.py` or another compatible decision/result file. For each decision row, the target is `+1` if the acting side eventually won, `-1` if it lost, and `0` for ties or unknown winners.

In [ ]:
TRAJECTORY_DATA_PATH = Path("training_data/trajectory_mcts.jsonl")
VALUE_CHECKPOINT_PATH = Path("checkpoints/value_mlp.pt")
VALUE_BATCH_SIZE = 64
VALUE_EPOCHS = 2 if SMOKE_RUN else 8
VALUE_LEARNING_RATE = 3e-4
VALUE_WEIGHT_DECAY = 1e-4
VALUE_VALIDATION_SPLIT = 0.1
VALUE_HIDDEN_SIZES = (512, 256)
VALUE_DROPOUT = 0.1

if not TRAJECTORY_DATA_PATH.exists():
    raise FileNotFoundError(
        f"{TRAJECTORY_DATA_PATH} not found. Generate a trajectory JSONL with "
        "collect_trajectory_mcts.py before training the value head."
    )

print("training value head from", TRAJECTORY_DATA_PATH.resolve())


In [ ]:
from dataclasses import asdict

from dataset import attach_results
from encode import EncoderConfig, encode_battle_state, feature_size
from torch import nn
from torch.utils.data import DataLoader, Dataset, random_split

value_encoder_config = EncoderConfig(hash_buckets=2048)
if CHECKPOINT_PATH.exists():
    policy_checkpoint = torch.load(CHECKPOINT_PATH, map_location="cpu")
    value_encoder_config = EncoderConfig(**policy_checkpoint["encoder_config"])

def value_target_from_record(record):
    metadata = dict(record.get("metadata") or {})
    winner = str(record.get("winner") or "").strip()
    actor = str(metadata.get("actor") or "").strip()
    opponent = str(metadata.get("opponent") or "").strip()
    if winner and actor and winner == actor:
        return 1.0
    if winner and opponent and winner == opponent:
        return -1.0
    return 0.0

class ValueJsonlDataset(Dataset):
    def __init__(self, path, encoder_config):
        self.path = Path(path)
        self.encoder_config = encoder_config
        self.records = []
        self.target_counts = {1.0: 0, -1.0: 0, 0.0: 0}
        for record in attach_results(self.path):
            metadata = dict(record.get("metadata") or {})
            if not isinstance(record.get("state"), dict):
                continue
            if not metadata.get("actor") or not metadata.get("opponent"):
                continue
            target = value_target_from_record(record)
            self.target_counts[target] = self.target_counts.get(target, 0) + 1
            self.records.append(record)
        if not self.records:
            raise ValueError(
                f"No decision rows with actor/opponent metadata were found in {self.path}"
            )
        nonzero_targets = self.target_counts.get(1.0, 0) + self.target_counts.get(-1.0, 0)
        if nonzero_targets == 0:
            raise ValueError(
                f"{self.path} has no resolved winners for value training; all targets are 0. "
                "Collect completed trajectory games with result rows before training the value head."
            )

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        record = self.records[index]
        state = dict(record["state"])
        state.setdefault("action_mask", record.get("action_mask") or [])
        encoded = encode_battle_state(state, config=self.encoder_config)
        target = value_target_from_record(record)
        return {
            "features": torch.tensor(encoded.features, dtype=torch.float32),
            "target": torch.tensor(target, dtype=torch.float32),
            "example_id": str(record.get("example_id") or ""),
        }

class ValueMLP(nn.Module):
    def __init__(self, input_dim, hidden_sizes=(512, 256), dropout=0.1):
        super().__init__()
        layers = []
        previous_dim = input_dim
        for hidden_dim in hidden_sizes:
            layers.extend([
                nn.Linear(previous_dim, hidden_dim),
                nn.ReLU(),
                nn.LayerNorm(hidden_dim),
                nn.Dropout(dropout),
            ])
            previous_dim = hidden_dim
        self.trunk = nn.Sequential(*layers)
        self.value_head = nn.Sequential(nn.Linear(previous_dim, 1), nn.Tanh())

    def forward(self, features):
        hidden = self.trunk(features)
        return self.value_head(hidden).squeeze(-1)

def run_value_epoch(model, loader, optimizer=None, device=None):
    is_training = optimizer is not None
    model.train(is_training)
    total_loss = 0.0
    total_sign_acc = 0.0
    total_examples = 0
    context = torch.enable_grad() if is_training else torch.no_grad()
    with context:
        for batch in loader:
            features = batch["features"].to(device)
            target = batch["target"].to(device)
            pred = model(features)
            loss = torch.mean((pred - target) ** 2)
            if is_training:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
            batch_size = features.shape[0]
            sign_acc = (torch.sign(pred) == torch.sign(target)).float().mean().item()
            total_loss += float(loss.item()) * batch_size
            total_sign_acc += sign_acc * batch_size
            total_examples += batch_size
    return {
        "loss": total_loss / max(1, total_examples),
        "sign_acc": total_sign_acc / max(1, total_examples),
        "examples": total_examples,
    }

value_dataset = ValueJsonlDataset(TRAJECTORY_DATA_PATH, encoder_config=value_encoder_config)
validation_size = max(1, int(round(len(value_dataset) * VALUE_VALIDATION_SPLIT))) if len(value_dataset) > 1 else 0
validation_size = min(validation_size, len(value_dataset) - 1) if len(value_dataset) > 1 else 0
train_size = len(value_dataset) - validation_size
if validation_size:
    value_train_dataset, value_validation_dataset = random_split(
        value_dataset,
        [train_size, validation_size],
        generator=torch.Generator().manual_seed(1337),
    )
else:
    value_train_dataset = value_dataset
    value_validation_dataset = value_dataset

value_train_loader = DataLoader(value_train_dataset, batch_size=VALUE_BATCH_SIZE, shuffle=True)
value_validation_loader = DataLoader(value_validation_dataset, batch_size=VALUE_BATCH_SIZE, shuffle=False)

value_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
value_model = ValueMLP(
    input_dim=feature_size(value_encoder_config),
    hidden_sizes=VALUE_HIDDEN_SIZES,
    dropout=VALUE_DROPOUT,
).to(value_device)
value_optimizer = torch.optim.AdamW(
    value_model.parameters(),
    lr=VALUE_LEARNING_RATE,
    weight_decay=VALUE_WEIGHT_DECAY,
)

best_value_checkpoint = None
best_value_loss = float("inf")
for epoch in range(1, VALUE_EPOCHS + 1):
    train_metrics = run_value_epoch(value_model, value_train_loader, optimizer=value_optimizer, device=value_device)
    validation_metrics = run_value_epoch(value_model, value_validation_loader, optimizer=None, device=value_device)
    print(
        f"epoch={epoch} train_loss={train_metrics['loss']:.4f} train_sign_acc={train_metrics['sign_acc']:.3f} "
        f"val_loss={validation_metrics['loss']:.4f} val_sign_acc={validation_metrics['sign_acc']:.3f}"
    )
    checkpoint = {
        "epoch": epoch,
        "model_state_dict": value_model.state_dict(),
        "optimizer_state_dict": value_optimizer.state_dict(),
        "model": {
            "class": "ValueMLP",
            "input_dim": feature_size(value_encoder_config),
            "hidden_sizes": VALUE_HIDDEN_SIZES,
            "dropout": VALUE_DROPOUT,
        },
        "encoder_config": asdict(value_encoder_config),
        "train_metrics": train_metrics,
        "validation_metrics": validation_metrics,
        "data_path": str(TRAJECTORY_DATA_PATH),
    }
    if validation_metrics["loss"] <= best_value_loss:
        best_value_loss = validation_metrics["loss"]
        best_value_checkpoint = checkpoint

VALUE_CHECKPOINT_PATH.parent.mkdir(parents=True, exist_ok=True)
torch.save(best_value_checkpoint, VALUE_CHECKPOINT_PATH)
print("saved", VALUE_CHECKPOINT_PATH.resolve())
print("value examples", len(value_dataset), "best val loss", round(best_value_loss, 4))
print("target counts", value_dataset.target_counts)


## 9. Verify The Value Checkpoint

In [ ]:
value_checkpoint = torch.load(VALUE_CHECKPOINT_PATH, map_location="cpu")
value_model = ValueMLP(
    input_dim=value_checkpoint["model"]["input_dim"],
    hidden_sizes=tuple(value_checkpoint["model"]["hidden_sizes"]),
    dropout=value_checkpoint["model"]["dropout"],
)
value_model.load_state_dict(value_checkpoint["model_state_dict"])
value_model.eval()
verify_encoder_config = EncoderConfig(**value_checkpoint["encoder_config"])

sample_records = []
for record in attach_results(TRAJECTORY_DATA_PATH):
    metadata = dict(record.get("metadata") or {})
    if metadata.get("actor") and metadata.get("opponent") and isinstance(record.get("state"), dict):
        sample_records.append(record)
    if len(sample_records) == 3:
        break

assert sample_records
for record in sample_records:
    encoded = encode_battle_state(record["state"], config=verify_encoder_config)
    features = torch.tensor(encoded.features, dtype=torch.float32).unsqueeze(0)
    pred = float(value_model(features).item())
    print(
        record.get("example_id"),
        "target",
        value_target_from_record(record),
        "pred",
        round(pred, 4),
    )


## 10. Scaling Up

For a larger Colab run, set `SMOKE_RUN=False`, then raise collection gradually: `POSITIONS=256`, `HYPOTHESES=4`, `MCTS_MS=75` is a reasonable next step. Resume training by setting `training.resume_checkpoint_path: checkpoints/policy_mlp.pt` in `configs/student.yaml` or by passing `--resume-checkpoint-path checkpoints/policy_mlp.pt` to `train.py`. For value training, point `TRAJECTORY_DATA_PATH` at a larger trajectory JSONL with decision/result rows and increase `VALUE_EPOCHS` once the dataset is large enough.